In [12]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), ".")))

from modules.data_loader import load_config, load_ground_truth, load_experiment_json
from modules.kbrs_metrics import get_official_metrics_df
from modules.visualization import feature

In [13]:
cfg = load_config()

In [14]:
fold1_test_replays = [275, 1725, 3613, 4520, 4664]

In [16]:
gt_df = load_ground_truth(cfg["data"]["label_dst_dir"], fold1_test_replays, "all_correct")

In [17]:
epoch = 30
label_method = "all_correct"

In [18]:
maskrcnn_win4_vanilla_df = load_experiment_json(cfg["data"]["predictions_dir"], cfg["experiments"]["maskrcnn"]["win4"]["vanilla"], epoch, label_method)
maskrcnn_win4_kbrs_df    = load_experiment_json(cfg["data"]["predictions_dir"], cfg["experiments"]["maskrcnn"]["win4"]["kbrs"], epoch, label_method)

In [19]:
result_gt_maskrcnn_win4_vanilla = get_official_metrics_df(gt_df, maskrcnn_win4_vanilla_df)
result_gt_maskrcnn_win4_kbrs    = get_official_metrics_df(gt_df, maskrcnn_win4_kbrs_df)

In [20]:
result_gt_maskrcnn_win4_vanilla.mean()

image_id    33340.944580
ir              0.734246
ic@000          0.829121
ic@030          0.803279
ic@050          0.781756
dtype: float64

In [21]:
result_gt_maskrcnn_win4_kbrs.mean()

image_id    33345.000000
ir              0.759439
ic@000          0.855305
ic@030          0.830322
ic@050          0.807858
dtype: float64

In [22]:
rtdetr_win1_kbrs_df      = load_experiment_json(cfg["data"]["predictions_dir"], cfg["experiments"]["rtdetr"]["win1"]["kbrs"], epoch, label_method)
rtdetr_win4_kbrs_df      = load_experiment_json(cfg["data"]["predictions_dir"], cfg["experiments"]["rtdetr"]["win4"]["kbrs"], epoch, label_method)

In [23]:
result_gt_rtdetr_win1_kbrs = get_official_metrics_df(gt_df, rtdetr_win1_kbrs_df)
result_gt_rtdetr_win4_kbrs = get_official_metrics_df(gt_df, rtdetr_win4_kbrs_df)

In [24]:
result_gt_rtdetr_win1_kbrs.mean()

image_id    33281.161577
ir              0.566112
ic@000          0.700859
ic@030          0.645691
ic@050          0.605635
dtype: float64

In [25]:
result_gt_rtdetr_win4_kbrs.mean()

image_id    33341.285794
ir              0.582373
ic@000          0.705161
ic@030          0.659517
ic@050          0.620382
dtype: float64

In [26]:
import numpy as np

def calculate_box_iou(box1, box2):
    """[x, y, w, h] 포맷의 두 박스 간의 일반적인 IoU를 계산합니다."""
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    
    ix1 = max(x1, x2)
    iy1 = max(y1, y2)
    ix2 = min(x1 + w1, x2 + w2)
    iy2 = min(y1 + h1, y2 + h2)
    
    inter_w = max(0, ix2 - ix1)
    inter_h = max(0, iy2 - iy1)
    inter_area = inter_w * inter_h
    
    union_area = (w1 * h1) + (w2 * h2) - inter_area
    return inter_area / union_area if union_area > 0 else 0

def get_mask_from_boxes(boxes, width=128, height=128):
    """박스 리스트를 받아 width x height 크기의 불리언 마스크를 생성합니다."""
    mask = np.zeros((height, width), dtype=bool)
    for box in boxes:
        x, y, w, h = box
        # 정수형 픽셀 인덱스로 변환 및 이미지 크기에 맞게 클리핑
        x_start, y_start = max(0, int(x)), max(0, int(y))
        x_end, y_end = min(width, int(x + w)), min(height, int(y + h))
        
        if x_start < x_end and y_start < y_end:
            mask[y_start:y_end, x_start:x_end] = True
    return mask

def get_multi_region_metrics_df(gt_df, pred_df, top_k=3, iou_threshold=0.3, width=128, height=128):
    """
    객체 단위(Object-level)와 영역 단위(Area-level) 다중 탐지 지표를 동시에 계산합니다.
    """
    pred_topk = pred_df.sort_values(by=['replay', 'image_id', 'score'], ascending=[True, True, False])
    pred_topk = pred_topk.groupby(['replay', 'image_id']).head(top_k)
    
    gt_grouped = gt_df.groupby(['replay', 'image_id'])['bbox'].apply(list).to_dict()
    pred_grouped = pred_topk.groupby(['replay', 'image_id'])['bbox'].apply(list).to_dict()
    
    results = []
    
    for (rep, iid), gts in gt_grouped.items():
        preds = pred_grouped.get((rep, iid), [])
        
        # 1. 예외 처리: GT나 Pred가 없는 경우
        if not preds or not gts:
            results.append({
                'replay': rep, 'image_id': iid,
                f'obj_recall@{top_k}': 0.0, f'obj_precision@{top_k}': 0.0,
                f'area_recall@{top_k}': 0.0, f'area_precision@{top_k}': 0.0, f'area_iou@{top_k}': 0.0
            })
            continue
            
        # 2. 객체 단위 (Object-level) 지표
        gt_hits = sum(1 for gt in gts if any(calculate_box_iou(gt, p) >= iou_threshold for p in preds))
        obj_recall = gt_hits / len(gts)
        
        pred_hits = sum(1 for p in preds if any(calculate_box_iou(gt, p) >= iou_threshold for gt in gts))
        obj_precision = pred_hits / len(preds)
        
        # 3. 영역 단위 (Area-level) 지표 (Boolean Masking)
        gt_mask = get_mask_from_boxes(gts, width, height)
        pred_mask = get_mask_from_boxes(preds, width, height)
        
        inter_area = np.logical_and(gt_mask, pred_mask).sum()
        gt_area = gt_mask.sum()
        pred_area = pred_mask.sum()
        union_area = np.logical_or(gt_mask, pred_mask).sum()
        
        area_recall = inter_area / gt_area if gt_area > 0 else 0.0
        area_precision = inter_area / pred_area if pred_area > 0 else 0.0
        area_iou = inter_area / union_area if union_area > 0 else 0.0
        
        results.append({
            'replay': rep,
            'image_id': iid,
            f'obj_recall@{top_k}': obj_recall,
            f'obj_precision@{top_k}': obj_precision,
            f'area_recall@{top_k}': area_recall,
            f'area_precision@{top_k}': area_precision,
            f'area_iou@{top_k}': area_iou
        })
        
    return pd.DataFrame(results)

In [27]:
# 상위 3개의 박스를 사용하는 다중 지역 지표 계산 (IoU 0.3 기준)
TOP_K = 3
vanilla_multi_df = get_multi_region_metrics_df(gt_df, maskrcnn_win4_vanilla_df, top_k=TOP_K, iou_threshold=0.3)
kbrs_multi_df = get_multi_region_metrics_df(gt_df, maskrcnn_win4_kbrs_df, top_k=TOP_K, iou_threshold=0.3)
rtdetr_win1_multi_df = get_multi_region_metrics_df(gt_df, rtdetr_win1_kbrs_df, top_k=TOP_K, iou_threshold=0.3)
rtdetr_win4_multi_df = get_multi_region_metrics_df(gt_df, rtdetr_win4_kbrs_df, top_k=TOP_K, iou_threshold=0.3)
# 평균 성능 출력


# print("\n[RTDETR Model - win 1]")
# print(f" - Recall@{TOP_K}: {rtdetr_win1_multi_df[f'recall@{TOP_K}'].mean():.4f}")
# print(f" - Precision@{TOP_K}: {rtdetr_win1_multi_df[f'precision@{TOP_K}'].mean():.4f}")

# print("\n[RTDETR Model - win 4]")
# print(f" - Recall@{TOP_K}: {rtdetr_win4_multi_df[f'recall@{TOP_K}'].mean():.4f}")
# print(f" - Precision@{TOP_K}: {rtdetr_win4_multi_df[f'precision@{TOP_K}'].mean():.4f}")

In [28]:
print(f"=== Detection Hit Ratio (Top-{TOP_K}) ===")
print(f" [MaskRCNN]        Recall: {vanilla_multi_df[f'obj_recall@{TOP_K}'].mean():.3f} | Precision: {vanilla_multi_df[f'obj_precision@{TOP_K}'].mean():.3f}")
print(f" [MaskRCNN + KBRS] Recall: {kbrs_multi_df[f'obj_recall@{TOP_K}'].mean():.3f} | Precision: {kbrs_multi_df[f'obj_precision@{TOP_K}'].mean():.3f}")
print(f" [RTDETRw1 + KBRS] Recall: {rtdetr_win1_multi_df[f'obj_recall@{TOP_K}'].mean():.3f} | Precision: {rtdetr_win1_multi_df[f'obj_precision@{TOP_K}'].mean():.3f}")
print(f" [RTDETRw4 + KBRS] Recall: {rtdetr_win4_multi_df[f'obj_recall@{TOP_K}'].mean():.3f} | Precision: {rtdetr_win4_multi_df[f'obj_precision@{TOP_K}'].mean():.3f}")

print(f"=== Region Coverage ===")
print(f" [MaskRCNN]        Area Recall: {vanilla_multi_df[f'area_recall@{TOP_K}'].mean():.3f} | Area IoU: {vanilla_multi_df[f'area_iou@{TOP_K}'].mean():.3f}")
print(f" [MaskRCNN + KBRS] Area Recall: {kbrs_multi_df[f'area_recall@{TOP_K}'].mean():.3f} | Area IoU: {kbrs_multi_df[f'area_iou@{TOP_K}'].mean():.3f}")
print(f" [RTDETRw1 + KBRS] Area Recall: {rtdetr_win1_multi_df[f'area_recall@{TOP_K}'].mean():.3f} | Area IoU: {rtdetr_win1_multi_df[f'area_iou@{TOP_K}'].mean():.3f}")
print(f" [RTDETRw4 + KBRS] Area Recall: {rtdetr_win4_multi_df[f'area_recall@{TOP_K}'].mean():.3f} | Area IoU: {rtdetr_win4_multi_df[f'area_iou@{TOP_K}'].mean():.3f}")

=== Detection Hit Ratio (Top-3) ===
 [MaskRCNN]        Recall: 0.586 | Precision: 0.590
 [MaskRCNN + KBRS] Recall: 0.465 | Precision: 0.743
 [RTDETRw1 + KBRS] Recall: 0.461 | Precision: 0.557
 [RTDETRw4 + KBRS] Recall: 0.465 | Precision: 0.560
=== Region Coverage ===
 [MaskRCNN]        Area Recall: 0.425 | Area IoU: 0.312
 [MaskRCNN + KBRS] Area Recall: 0.325 | Area IoU: 0.293
 [RTDETRw1 + KBRS] Area Recall: 0.340 | Area IoU: 0.265
 [RTDETRw4 + KBRS] Area Recall: 0.346 | Area IoU: 0.270


In [29]:
import math
from itertools import combinations

def _get_box_centroid(box):
    """[x, y, w, h] 박스의 중심점 (cx, cy)를 반환합니다."""
    return box[0] + box[2] / 2.0, box[1] + box[3] / 2.0

def get_gt_spatial_stats(gts):
    """
    GT 박스들 간의 최대 이격 거리(max_dist)와 최대 겹침 정도(max_iou)를 계산합니다.
    """
    if len(gts) < 2:
        return 0.0, 0.0
        
    max_dist = 0.0
    max_iou = 0.0
    
    for b1, b2 in combinations(gts, 2):
        # 거리 계산
        cx1, cy1 = _get_box_centroid(b1)
        cx2, cy2 = _get_box_centroid(b2)
        dist = math.hypot(cx1 - cx2, cy1 - cy2)
        if dist > max_dist:
            max_dist = dist
            
        # IoU 계산
        iou = calculate_box_iou(b1, b2)
        if iou > max_iou:
            max_iou = iou
            
    return max_dist, max_iou

def get_rtdetr_advantage_metrics(gt_df, pred_df, top_k=3, iou_threshold=0.3):
    """
    프레임의 공간적 특성(거리, 겹침) 메타데이터가 포함된 성능 지표를 반환합니다.
    """
    pred_topk = pred_df.sort_values(by=['replay', 'image_id', 'score'], ascending=[True, True, False])
    pred_topk = pred_topk.groupby(['replay', 'image_id']).head(top_k)
    
    gt_grouped = gt_df.groupby(['replay', 'image_id'])['bbox'].apply(list).to_dict()
    pred_grouped = pred_topk.groupby(['replay', 'image_id'])['bbox'].apply(list).to_dict()
    
    results = []
    
    for (rep, iid), gts in gt_grouped.items():
        preds = pred_grouped.get((rep, iid), [])
        
        # GT가 2개 이상인 '다중 교전' 상황만 분석 대상
        if len(gts) < 2:
            continue
            
        max_dist, max_gt_iou = get_gt_spatial_stats(gts)
        
        # 기본 객체 단위 지표 계산
        if not preds:
            recall, precision = 0.0, 0.0
        else:
            gt_hits = sum(1 for gt in gts if any(calculate_box_iou(gt, p) >= iou_threshold for p in preds))
            recall = gt_hits / len(gts)
            
            pred_hits = sum(1 for p in preds if any(calculate_box_iou(gt, p) >= iou_threshold for gt in gts))
            precision = pred_hits / len(preds)
            
        results.append({
            'replay': rep,
            'image_id': iid,
            'gt_count': len(gts),
            'max_dist': max_dist,       # 양동 작전(Multi-front) 판별용
            'max_gt_iou': max_gt_iou,   # 인접 교전(Clustered) 판별용
            f'recall@{top_k}': recall,
            f'precision@{top_k}': precision
        })
        
    return pd.DataFrame(results)

In [30]:
a = get_rtdetr_advantage_metrics(gt_df, maskrcnn_win4_vanilla_df, top_k=TOP_K)
b = get_rtdetr_advantage_metrics(gt_df, maskrcnn_win4_kbrs_df, top_k=TOP_K)
c = get_rtdetr_advantage_metrics(gt_df, rtdetr_win1_kbrs_df, top_k=TOP_K)
d = get_rtdetr_advantage_metrics(gt_df, rtdetr_win4_kbrs_df, top_k=TOP_K)

In [31]:
DIST_THRESH = 64.0 

print("=== 1. Multi-region coverage (GT Distance >= 64) ===")
for name, df in zip(['MaskRCNN', 'MaskRCNN+KBRS', 'RT-DETRw1', 'RT-DETRw4'], [a, b, c, d]):
    subset = df[df['max_dist'] >= DIST_THRESH]
    if len(subset) > 0:
        print(f"[{name:<14s}] Recall@{TOP_K}: {subset[f'recall@{TOP_K}'].mean():.3f}")

# ---------------------------------------------------------
# 지표 4: 인접 교전 분리 능력 (NMS-Free Advantage)
# ---------------------------------------------------------
# 조건: GT 박스끼리 겹치는 영역이 있는(IoU > 0.05) 매우 밀집된 복합 교전 상황
OVERLAP_THRESH = 0.05

print("\n=== 2. NMS (GT IoU >= 0.05) ===")
for name, df in zip(['MaskRCNN', 'MaskRCNN+KBRS', 'RT-DETRw1', 'RT-DETRw4'], [a, b, c, d]):
    subset = df[df['max_gt_iou'] >= OVERLAP_THRESH]
    if len(subset) > 0:
        print(f"[{name:<14s}] Precision@{TOP_K}: {subset[f'precision@{TOP_K}'].mean():.3f}")

=== 1. Multi-region coverage (GT Distance >= 64) ===
[MaskRCNN      ] Recall@3: 0.538
[MaskRCNN+KBRS ] Recall@3: 0.396
[RT-DETRw1     ] Recall@3: 0.422
[RT-DETRw4     ] Recall@3: 0.422

=== 2. NMS (GT IoU >= 0.05) ===
[MaskRCNN      ] Precision@3: 0.593
[MaskRCNN+KBRS ] Precision@3: 0.751
[RT-DETRw1     ] Precision@3: 0.558
[RT-DETRw4     ] Precision@3: 0.563
